# 8.3 ?? ??? ?? ???? ????? ??

- [?? ??? ?? ???? ??](https://www.kaggle.com/c/porto-seguro-safe-driver-prediction)
- [????? ?? ?? ??](https://www.kaggle.com/xiaozhouwang/2nd-place-lightgbm-solution)

In [ ]:
import pandas as pd

# ??? ??
data_path = '/kaggle/input/porto-seguro-safe-driver-prediction/'

train = pd.read_csv(data_path + 'train.csv', index_col='id')
test = pd.read_csv(data_path + 'test.csv', index_col='id')
submission = pd.read_csv(data_path + 'sample_submission.csv', index_col='id')

## 8.3.1 ?? ?????

### ??? ???

In [ ]:
all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.drop('target', axis=1) # ??? ??

In [ ]:
all_features = all_data.columns # ?? ??
all_features

### ??? ?? ?-? ???

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# ??? ?? ??
cat_features = [feature for feature in all_features if 'cat' in feature] 

onehot_encoder = OneHotEncoder() # ?-? ??? ?? ??
# ???
encoded_cat_matrix = onehot_encoder.fit_transform(all_data[cat_features]) 

encoded_cat_matrix

### ?? ?? ?? ??

In [ ]:
# ??? ??? ??
drop_features = ['ps_ind_14', 'ps_ind_10_bin', 'ps_ind_11_bin', 
                 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_car_14']

# '1) ??? ??, 2) calc ??? ??, 3) ?? ??? ??'? ??? ??
remaining_features = [feature for feature in all_features 
                      if ('cat' not in feature and 
                          'calc' not in feature and 
                          feature not in drop_features)]

In [ ]:
from scipy import sparse

all_data_sprs = sparse.hstack([sparse.csr_matrix(all_data[remaining_features]),
                               encoded_cat_matrix],
                              format='csr')

### ??? ???

In [ ]:
num_train = len(train) # ?? ??? ??

# ?? ???? ??? ??? ???
X = all_data_sprs[:num_train]
X_test = all_data_sprs[num_train:]

y = train['target'].values

## 8.3.2 ???? ?? ?? ??

### ??? ???? ?? ??

In [ ]:
import numpy as np

def eval_gini(y_true, y_pred):
    # ???? ???? ??? ??? ?? (?? ??? ?? ??)
    assert y_true.shape == y_pred.shape

    n_samples = y_true.shape[0]                      # ??? ??
    L_mid = np.linspace(1 / n_samples, 1, n_samples) # ??? ?

    # 1) ???? ?? ????
    pred_order = y_true[y_pred.argsort()] # y_pred ????? y_true ? ??
    L_pred = np.cumsum(pred_order) / np.sum(pred_order) # ??? ??
    G_pred = np.sum(L_mid - L_pred)       # ?? ?? ?? ????

    # 2) ??? ??? ? ????
    true_order = y_true[y_true.argsort()] # y_true ????? y_true ? ??
    L_true = np.cumsum(true_order) / np.sum(true_order) # ??? ??
    G_true = np.sum(L_mid - L_true)       # ??? ??? ? ????

    # ???? ????
    return G_pred / G_true

In [ ]:
def gini(preds, dtrain):
    labels = dtrain.get_label()
    return 'gini', eval_gini(labels, preds), True # ???

## 8.3.3 ?? ?? ? ?? ??

### OOF ???? LightGBM ??

In [ ]:
from sklearn.model_selection import StratifiedKFold

# ?? K ?? ?? ???
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=1991)

In [ ]:
params = {'objective': 'binary',
          'learning_rate': 0.01,
          'force_row_wise': True,
          'random_state': 0}

In [ ]:
# OOF ???? ??? ??? ?? ??? ???? ??? ??? ?? 1?? ??
oof_val_preds = np.zeros(X.shape[0]) 
# OOF ???? ??? ??? ??? ??? ???? ??? ??? ?? 1?? ??
oof_test_preds = np.zeros(X_test.shape[0]) 

In [ ]:
import lightgbm as lgb

# OOF ???? ?? ??, ??, ??
for idx, (train_idx, valid_idx) in enumerate(folds.split(X, y)):
    # ? ??? ???? ?? ??
    print('#'*40, f'?? {idx+1} / ?? {folds.n_splits}', '#'*40)
    
    # ??? ???, ??? ??? ?? 
    X_train, y_train = X[train_idx], y[train_idx] # ??? ???
    X_valid, y_valid = X[valid_idx], y[valid_idx] # ??? ???

    # LightGBM ?? ???? ?? 
    dtrain = lgb.Dataset(X_train, y_train) # LightGBM ?? ?? ????
    dvalid = lgb.Dataset(X_valid, y_valid) # LightGBM ?? ?? ????

    # LightGBM ?? ?? 
    lgb_model = lgb.train(params=params,        # ??? ???????
                          train_set=dtrain,     # ?? ????
                          num_boost_round=1000, # ??? ?? ??
                          valid_sets=dvalid,    # ?? ??? ?? ????
                          feval=gini,           # ??? ????
                          early_stopping_rounds=100, # ???? ??
                          verbose_eval=100)     # 100???? ?? ??
    
    # ??? ???? ??? OOF ??
    oof_test_preds += lgb_model.predict(X_test)/folds.n_splits
    
    # ?? ?? ??? ?? ?? ??? ??? ??
    oof_val_preds[valid_idx] += lgb_model.predict(X_valid)
    
    # ?? ??? ?? ??? ?? ??? ???? 
    gini_score = eval_gini(y_valid, oof_val_preds[valid_idx])
    print(f'?? {idx+1} ???? : {gini_score}\n')

In [ ]:
print('OOF ?? ??? ????:', eval_gini(y, oof_val_preds))

## 8.3.4 ?? ? ?? ??

In [ ]:
submission['target'] = oof_test_preds
submission.to_csv('submission.csv')